# AAI 540 semantic retrieval and grounded Product QA inference

Stage 5 completes the core user outcome: a caller supplies a request ID, product ASIN, and English question; the system retrieves product evidence using MiniLM embeddings and FAISS, reranks candidates with the trained classifier, then returns an extractive answer with ranked evidence or abstains when relevance confidence is low.

No free-form answer generation is used. The returned answer is always copied from retrieved evidence, keeping the MVP grounded and auditable.

## Prerequisites and cost boundary

Run stages 1-4 first. This notebook downloads the approved data/model lineage, builds the embedding index in the notebook kernel, and uploads one portable runtime artifact. It creates no endpoint. The first MiniLM load downloads public model weights; subsequent loads use the local cache.

In [1]:
%pip install -q --upgrade \
    "boto3>=1.34" \
    "botocore>=1.34" \
    "pandas>=2.1,<3" \
    "pyarrow>=15,<22" \
    "scikit-learn==1.4.2" \
    "joblib>=1.3,<2" \
    "sentence-transformers>=3.0,<4" \
    "faiss-cpu>=1.8,<2" \
    "torch>=2.1,<3"

Note: you may need to restart the kernel to use updated packages.


> Restart the kernel only if an already-loaded dependency could not be upgraded, then continue from the import cell.

In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

print("Configured Transformers for PyTorch only")

Configured Transformers for PyTorch only


In [9]:
%pip install -q "pyarrow==20.0.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sagemaker-studio 1.1.32 requires pymongo-auth-aws>=1.3.0, which is not installed.
sagemaker-studio 1.1.32 requires pymongosql>=0.7.2, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import importlib.util
import pyarrow as pa

print("PyArrow:", pa.__version__)
print("PyExtensionType available:", hasattr(pa, "PyExtensionType"))
print("Torchvision:", importlib.util.find_spec("torchvision"))

assert pa.__version__ == "20.0.0"
assert hasattr(pa, "PyExtensionType")
assert importlib.util.find_spec("torchvision") is None

PyArrow: 20.0.0
PyExtensionType available: True
Torchvision: None


In [2]:
import importlib.util

assert importlib.util.find_spec("torchvision") is None

from sentence_transformers import SentenceTransformer

print("SentenceTransformers PyTorch import passed")

SentenceTransformers PyTorch import passed


In [3]:
import json
import os
import tarfile
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd
from botocore.config import Config
from IPython.display import display

from inference_code.retrieval import (
    RetrievalQAEngine,
    build_artifacts,
    build_corpus,
    make_candidate_id,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

## 1 Configuration and AWS preflight

In [4]:
# ---------- User-editable settings ----------
PROJECT_NAME = "product-qa"
AWS_REGION_OVERRIDE = ""
S3_BUCKET_OVERRIDE = ""
MODEL_ARTIFACT_OVERRIDE = ""            # Optional approved/champion model.tar.gz URI.
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 128
MAX_CORPUS_ROWS = None                   # Keep None for the complete curated subset.
RETRIEVAL_K = 50
EVIDENCE_K = 3
MINIMUM_PROBABILITY_2 = 0.35
KNOWN_QUERY_SAMPLE_SIZE = 50
MINIMUM_KNOWN_QUERY_RECALL_AT_3 = 0.40
# -------------------------------------------

session = boto3.Session(region_name=AWS_REGION_OVERRIDE or None)
region = session.region_name or os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")
if not region:
    raise RuntimeError("No AWS Region is configured. Set AWS_REGION_OVERRIDE.")
retry_config = Config(retries={"max_attempts": 10, "mode": "adaptive"})
s3 = session.client("s3", config=retry_config)
sts = session.client("sts")
identity = sts.get_caller_identity()
account_id = identity["Account"]
bucket = S3_BUCKET_OVERRIDE or f"sagemaker-{region}-{account_id}"
retrieval_run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
retrieval_prefix = f"aai540/{PROJECT_NAME}/retrieval/{retrieval_run_id}"
local_dir = Path("/tmp") / f"aai540_retrieval_{retrieval_run_id}"
local_dir.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "region": region,
    "caller_arn": identity["Arn"],
    "bucket": bucket,
    "retrieval_run_id": retrieval_run_id,
    "embedding_model": EMBEDDING_MODEL_ID,
}, indent=2))

{
  "region": "us-east-1",
  "caller_arn": "arn:aws:sts::070881885726:assumed-role/LabRole/SageMaker",
  "bucket": "sagemaker-us-east-1-070881885726",
  "retrieval_run_id": "20261017_033939",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
}


## 2 Resolve the immutable data and model lineage

In [5]:
def split_s3_uri(uri):
    if not uri.startswith("s3://"):
        raise ValueError(f"Expected an S3 URI, got {uri}")
    bucket_name, _, key = uri[5:].partition("/")
    return bucket_name, key

def read_s3_json(uri):
    source_bucket, source_key = split_s3_uri(uri)
    return json.loads(s3.get_object(Bucket=source_bucket, Key=source_key)["Body"].read())

def download_s3_file(uri, destination):
    source_bucket, source_key = split_s3_uri(uri)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(source_bucket, source_key, str(destination))
    return destination

def download_s3_prefix(uri, destination):
    source_bucket, prefix = split_s3_uri(uri)
    destination = Path(destination)
    downloaded = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=source_bucket, Prefix=prefix):
        for item in page.get("Contents", []):
            if item["Key"].endswith("/"):
                continue
            relative = item["Key"][len(prefix):].lstrip("/") or Path(item["Key"]).name
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(source_bucket, item["Key"], str(target))
            downloaded.append(target)
    if not downloaded:
        raise FileNotFoundError(f"No objects found below {uri}")
    return downloaded

monitoring_receipt_uri = f"s3://{bucket}/aai540/{PROJECT_NAME}/monitoring/latest.json"
continual_receipt_uri = f"s3://{bucket}/aai540/{PROJECT_NAME}/continual-learning/latest.json"
monitoring_receipt = read_s3_json(monitoring_receipt_uri)
continual_receipt = read_s3_json(continual_receipt_uri)
modeling_receipt = read_s3_json(monitoring_receipt["modeling_receipt"])
run_summary = read_s3_json(modeling_receipt["source_run_summary"])

if continual_receipt.get("lineage", {}).get("monitoring_receipt") != monitoring_receipt_uri:
    raise ValueError(
        "The latest continual-learning receipt is not based on the latest monitoring receipt. "
        "Run stages 3 and 4 again before building retrieval artifacts."
    )
if continual_receipt.get("champion", {}).get("model_run_id") != modeling_receipt["model_run_id"]:
    raise ValueError("The continual-learning and modeling receipts disagree on the champion model")

continual_candidate = continual_receipt.get("candidate", {})
approved_continual_artifact = (
    continual_candidate.get("model_artifact")
    if continual_candidate.get("approval_status") == "Approved"
    else None
)
if MODEL_ARTIFACT_OVERRIDE:
    model_artifact_uri = MODEL_ARTIFACT_OVERRIDE
    source_model_role = "explicit approved/champion override"
elif approved_continual_artifact:
    model_artifact_uri = approved_continual_artifact
    source_model_role = "approved continual-learning candidate"
else:
    model_artifact_uri = modeling_receipt["candidate"]["model_artifact"]
    source_model_role = "existing champion"
curated_data_uri = run_summary["curated_data"]

lineage = {
    "source_data_run_id": run_summary["run_id"],
    "source_run_summary": modeling_receipt["source_run_summary"],
    "source_model_run_id": modeling_receipt["model_run_id"],
    "source_model_artifact": model_artifact_uri,
    "source_model_role": source_model_role,
    "monitor_run_id": monitoring_receipt["monitor_run_id"],
    "monitoring_receipt": monitoring_receipt_uri,
    "continual_learning_run_id": continual_receipt["continual_run_id"],
    "continual_learning_receipt": continual_receipt_uri,
    "continual_candidate_approval_status": continual_receipt["candidate"]["approval_status"],
    "curated_data": curated_data_uri,
}
print(json.dumps(lineage, indent=2))

{
  "source_data_run_id": "20261017_011905",
  "source_run_summary": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/runs/20261017_011905/run_summary.json",
  "source_model_run_id": "20261017_012600",
  "source_model_artifact": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/modeling/20261017_012600/artifacts/candidate/aai540-product-qa-candidate-2026-10-17-01-29-09-129/output/model.tar.gz",
  "source_model_role": "existing champion",
  "monitor_run_id": "20261017_014230",
  "monitoring_receipt": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/monitoring/latest.json",
  "continual_learning_run_id": "20261017_015154",
  "continual_learning_receipt": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/continual-learning/latest.json",
  "continual_candidate_approval_status": "PendingManualApproval",
  "curated_data": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/runs/20261017_011905/curated/epqa/"
}


## 3 Build the product evidence corpus

In [6]:
curated_files = download_s3_prefix(curated_data_uri, local_dir / "curated")
parquet_files = [path for path in curated_files if path.suffix.lower() == ".parquet"]
if not parquet_files:
    raise FileNotFoundError("The curated S3 prefix contains no Parquet files")
curated = pd.concat([pd.read_parquet(path) for path in parquet_files], ignore_index=True)

model_archive = download_s3_file(model_artifact_uri, local_dir / "source-model.tar.gz")
with tarfile.open(model_archive, "r:gz") as archive:
    member = archive.getmember("model.joblib")
    extracted = archive.extractfile(member)
    if extracted is None:
        raise FileNotFoundError("model.joblib is missing from the model artifact")
    source_model_path = local_dir / "source-model.joblib"
    source_model_path.write_bytes(extracted.read())
model_bundle = joblib.load(source_model_path)

corpus = build_corpus(curated)
if MAX_CORPUS_ROWS is not None and len(corpus) > MAX_CORPUS_ROWS:
    corpus = corpus.sort_values("candidate_id").head(MAX_CORPUS_ROWS).reset_index(drop=True)

display(pd.DataFrame({
    "metric": ["curated rows", "candidate corpus rows", "unique products", "sources"],
    "value": [len(curated), len(corpus), corpus["asin"].nunique(), corpus["source"].nunique()],
}))
display(corpus.head())

,metric,value
0,curated rows,24966
1,candidate corpus rows,24711
2,unique products,2128
3,sources,8


,candidate_id,asin,candidate,title,source
0,5e7e0000f2cc1fadcc18e787ce4dc3de,B00E37TR5U,"time will tell how long it lasts, the original filter from the manufacturer was about 6 months","EcoAqua EFF-6027A Replacement Filter, Compatible with Samsung DA29-00020B, DA29-00020A, HAF-CIN/EXP, 46-9101 Refrigerator Water Filter",review
1,5d8aeca1339890bb07c898bf1aaf1f48,B00E37TR5U,they are supposed to last around 6 months!,"EcoAqua EFF-6027A Replacement Filter, Compatible with Samsung DA29-00020B, DA29-00020A, HAF-CIN/EXP, 46-9101 Refrigerator Water Filter",review
2,4ee5d9f651fcccbb355850f940fd286f,B00E37TR5U,we are a family of 2 adults and 3 children and this filter lasts us about 6 months.,"EcoAqua EFF-6027A Replacement Filter, Compatible with Samsung DA29-00020B, DA29-00020A, HAF-CIN/EXP, 46-9101 Refrigerator Water Filter",review
3,372ef4d5a35950ec01f14cccef07109a,B00E37TR5U,it needs to be changed in 3 months no matter how often and how much water is used.,"EcoAqua EFF-6027A Replacement Filter, Compatible with Samsung DA29-00020B, DA29-00020A, HAF-CIN/EXP, 46-9101 Refrigerator Water Filter",review
4,43cdcdeeb8e4ba2b41aa0f6b0c194f9d,B00E37TR5U,this is supposed to be a 6 month filter.,"EcoAqua EFF-6027A Replacement Filter, Compatible with Samsung DA29-00020B, DA29-00020A, HAF-CIN/EXP, 46-9101 Refrigerator Water Filter",review


## 4 Generate MiniLM embeddings and the FAISS index

In [7]:
%pip uninstall -y torchvision

Found existing installation: torchvision 0.24.0
Uninstalling torchvision-0.24.0:
  Successfully uninstalled torchvision-0.24.0
Note: you may need to restart the kernel to use updated packages.


In [7]:
runtime_dir = local_dir / "runtime"
manifest = build_artifacts(
    corpus,
    model_bundle,
    runtime_dir,
    embedding_model_id=EMBEDDING_MODEL_ID,
    batch_size=EMBEDDING_BATCH_SIZE,
)
required_runtime_files = [
    runtime_dir / "model.joblib",
    runtime_dir / "corpus.parquet",
    runtime_dir / "embeddings.npy",
    runtime_dir / "faiss.index",
    runtime_dir / "retrieval_manifest.json",
    runtime_dir / "embedding_model",
]
missing_runtime_files = [str(path) for path in required_runtime_files if not path.exists()]
if missing_runtime_files:
    raise FileNotFoundError(f"Runtime build is incomplete: {missing_runtime_files}")
print(json.dumps(manifest, indent=2))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/194 [00:00<?, ?it/s]

{
  "schema_version": "aai540-retrieval-runtime-v1",
  "embedding_model_id": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dimensions": 384,
  "corpus_rows": 24711,
  "unique_products": 2128,
  "ranker_schema_version": "epqa-scalar-features-v1",
  "ranker_feature_columns": [
    "question_char_count",
    "candidate_char_count",
    "title_char_count",
    "hash_cosine_similarity",
    "question_word_count",
    "candidate_word_count",
    "title_word_count",
    "token_overlap_count",
    "token_jaccard",
    "number_overlap_count",
    "question_has_number",
    "candidate_has_number",
    "candidate_in_title",
    "missing_title"
  ],
  "retrieval": "FAISS inner-product search over L2-normalized MiniLM embeddings",
  "answer_policy": "Return the highest reranked extractive candidate or abstain"
}


## 5 Exercise the complete query-to-answer path

In [8]:
engine = RetrievalQAEngine.from_model_dir(
    runtime_dir,
    retrieval_k=RETRIEVAL_K,
    evidence_k=EVIDENCE_K,
    minimum_probability_2=MINIMUM_PROBABILITY_2,
)

demo_row = curated[curated["asin"].astype(str).ne("")].iloc[0]
demo_request = {
    "request_id": f"demo-{retrieval_run_id}",
    "asin": str(demo_row["asin"]),
    "question": str(demo_row["question"]),
}
started = time.perf_counter()
demo_response = engine.answer(demo_request)
demo_latency_ms = (time.perf_counter() - started) * 1000
print(json.dumps(demo_response, indent=2))
print(f"In-kernel latency: {demo_latency_ms:.1f} ms")

assert demo_response["request_id"] == demo_request["request_id"]
assert demo_response["asin"] == demo_request["asin"]
assert isinstance(demo_response["evidence"], list)
if not demo_response["abstained"]:
    assert demo_response["answer"] == demo_response["evidence"][0]["text"]

{
  "request_id": "demo-20261017_033939",
  "asin": "B00E37TR5U",
  "answer": "this is supposed to be a 6 month filter.",
  "abstained": false,
  "abstention_reason": null,
  "evidence": [
    {
      "rank": 1,
      "candidate_id": "43cdcdeeb8e4ba2b41aa0f6b0c194f9d",
      "text": "this is supposed to be a 6 month filter.",
      "title": "EcoAqua EFF-6027A Replacement Filter, Compatible with Samsung DA29-00020B, DA29-00020A, HAF-CIN/EXP, 46-9101 Refrigerator Water Filter",
      "source": "review",
      "semantic_similarity": 0.2779577374458313,
      "rerank_score": 0.586421284189977,
      "probability_0": 0.1645110340481134,
      "probability_1": 0.29249299902772163,
      "probability_2": 0.5429959669241647
    },
    {
      "rank": 2,
      "candidate_id": "5e7e0000f2cc1fadcc18e787ce4dc3de",
      "text": "time will tell how long it lasts, the original filter from the manufacturer was about 6 months",
      "title": "EcoAqua EFF-6027A Replacement Filter, Compatible with Sams

## 6 Known-query retrieval and grounding evaluation

In [9]:
positive = curated[
    curated["label"].astype(int).eq(2)
    & curated["asin"].fillna("").astype(str).ne("")
].drop_duplicates(["qid", "asin"]).copy()
sample_size = min(KNOWN_QUERY_SAMPLE_SIZE, len(positive))
if sample_size == 0:
    raise ValueError("No fully relevant labeled queries are available for the known-query test")
known_queries = positive.sample(n=sample_size, random_state=540)

evaluation_rows = []
for index, row in enumerate(known_queries.itertuples(index=False)):
    request_payload = {
        "request_id": f"known-{index}",
        "asin": str(row.asin),
        "question": str(row.question),
    }
    started = time.perf_counter()
    response = engine.answer(request_payload)
    latency_ms = (time.perf_counter() - started) * 1000
    expected_id = make_candidate_id(row.asin, row.candidate, row.title, row.source)
    returned_ids = [item["candidate_id"] for item in response["evidence"]]
    evaluation_rows.append({
        "request_id": request_payload["request_id"],
        "asin": request_payload["asin"],
        "expected_candidate_recalled_at_3": expected_id in returned_ids,
        "answered": not response["abstained"],
        "grounded": response["abstained"] or response["answer"] in [
            item["text"] for item in response["evidence"]
        ],
        "latency_ms": latency_ms,
    })

known_query_results = pd.DataFrame(evaluation_rows)
known_query_metrics = {
    "sample_rows": int(len(known_query_results)),
    "recall_at_3": float(known_query_results["expected_candidate_recalled_at_3"].mean()),
    "answer_rate": float(known_query_results["answered"].mean()),
    "grounded_response_rate": float(known_query_results["grounded"].mean()),
    "p50_latency_ms": float(known_query_results["latency_ms"].quantile(0.50)),
    "p95_latency_ms": float(known_query_results["latency_ms"].quantile(0.95)),
}
display(known_query_results.head(10))
print(json.dumps(known_query_metrics, indent=2))

if known_query_metrics["recall_at_3"] < MINIMUM_KNOWN_QUERY_RECALL_AT_3:
    raise RuntimeError(
        f"Known-query recall@3 {known_query_metrics['recall_at_3']:.3f} is below "
        f"{MINIMUM_KNOWN_QUERY_RECALL_AT_3:.3f}; do not package this index."
    )
if known_query_metrics["grounded_response_rate"] != 1.0:
    raise RuntimeError("At least one non-abstained answer was not copied from returned evidence")

,request_id,asin,expected_candidate_recalled_at_3,answered,grounded,latency_ms
0,known-0,B009LRNT1A,True,True,True,92.210877
1,known-1,B005VIWJUU,True,True,True,85.091802
2,known-2,B000KL72VY,False,True,True,81.818238
3,known-3,B001LU1SFO,False,True,True,93.586534
4,known-4,B0086I5YAE,True,True,True,91.033524
5,known-5,B001L4JH5I,True,True,True,83.264508
6,known-6,B0031XXW4O,False,True,True,84.180070
7,known-7,B0071FAKZM,True,True,True,93.475686
8,known-8,B003P443C4,True,True,True,83.838241
9,known-9,B00D84DRNC,False,True,True,87.943214


{
  "sample_rows": 50,
  "recall_at_3": 0.54,
  "answer_rate": 0.98,
  "grounded_response_rate": 1.0,
  "p50_latency_ms": 84.00841950060567,
  "p95_latency_ms": 97.52122660056556
}


## 7 Package and publish the portable inference runtime

In [10]:
model_card = f"""# Product QA retrieval runtime

- Retrieval run: {retrieval_run_id}
- Data run: {run_summary['run_id']}
- Ranker run: {modeling_receipt['model_run_id']}
- Ranker source role: {source_model_role}
- Continual-learning run: {continual_receipt['continual_run_id']}
- Embedding model: {EMBEDDING_MODEL_ID}
- Corpus rows: {manifest['corpus_rows']}
- Known-query recall@3: {known_query_metrics['recall_at_3']:.4f}
- Grounded response rate: {known_query_metrics['grounded_response_rate']:.4f}

The system returns extractive evidence only and abstains below the configured fully-relevant probability threshold. Evaluation uses a labeled subset of ePQA and is not evidence of performance on all products, languages, or question types.
"""
(runtime_dir / "MODEL_CARD.md").write_text(model_card, encoding="utf-8")

runtime_archive = local_dir / "product-qa-runtime.tar.gz"
with tarfile.open(runtime_archive, "w:gz") as archive:
    for path in sorted(runtime_dir.rglob("*")):
        if path.is_file():
            archive.add(path, arcname=str(path.relative_to(runtime_dir)))

artifact_key = f"{retrieval_prefix}/artifacts/product-qa-runtime.tar.gz"
s3.upload_file(str(runtime_archive), bucket, artifact_key)
runtime_artifact_uri = f"s3://{bucket}/{artifact_key}"

evaluation_key = f"{retrieval_prefix}/evaluation/known_query_results.csv"
evaluation_path = local_dir / "known_query_results.csv"
known_query_results.to_csv(evaluation_path, index=False)
s3.upload_file(str(evaluation_path), bucket, evaluation_key)

print(json.dumps({
    "runtime_artifact": runtime_artifact_uri,
    "artifact_bytes": runtime_archive.stat().st_size,
    "evaluation": f"s3://{bucket}/{evaluation_key}",
}, indent=2))

{
  "runtime_artifact": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/retrieval/20261017_033939/artifacts/product-qa-runtime.tar.gz",
  "artifact_bytes": 167278967,
  "evaluation": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/retrieval/20261017_033939/evaluation/known_query_results.csv"
}


## 8 Persist the retrieval receipt

In [11]:
retrieval_receipt = {
    "schema_version": "aai540-retrieval-receipt-v1",
    "project": PROJECT_NAME,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "retrieval_run_id": retrieval_run_id,
    "lineage": lineage,
    "runtime": {
        "artifact": runtime_artifact_uri,
        "embedding_model_id": EMBEDDING_MODEL_ID,
        "corpus_rows": manifest["corpus_rows"],
        "unique_products": manifest["unique_products"],
        "retrieval_k": RETRIEVAL_K,
        "evidence_k": EVIDENCE_K,
        "minimum_probability_2": MINIMUM_PROBABILITY_2,
    },
    "evaluation": {
        **known_query_metrics,
        "details": f"s3://{bucket}/{evaluation_key}",
        "minimum_recall_at_3": MINIMUM_KNOWN_QUERY_RECALL_AT_3,
        "passed": True,
    },
    "limitations": [
        "English ePQA subset only",
        "Extractive answers only",
        "No guarantee for products absent from the indexed corpus",
        "The random-forest reranker is not a calibrated language model",
    ],
}
receipt_body = json.dumps(retrieval_receipt, indent=2, allow_nan=False).encode("utf-8")
receipt_key = f"{retrieval_prefix}/retrieval_receipt.json"
latest_key = f"aai540/{PROJECT_NAME}/retrieval/latest.json"
for key in [receipt_key, latest_key]:
    s3.put_object(Bucket=bucket, Key=key, Body=receipt_body, ContentType="application/json")

print(json.dumps(retrieval_receipt, indent=2))
print(f"Retrieval receipt: s3://{bucket}/{receipt_key}")
print(f"Latest pointer: s3://{bucket}/{latest_key}")

{
  "schema_version": "aai540-retrieval-receipt-v1",
  "project": "product-qa",
  "created_at_utc": "2026-10-17T03:51:07.558158+00:00",
  "retrieval_run_id": "20261017_033939",
  "lineage": {
    "source_data_run_id": "20261017_011905",
    "source_run_summary": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/runs/20261017_011905/run_summary.json",
    "source_model_run_id": "20261017_012600",
    "source_model_artifact": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/modeling/20261017_012600/artifacts/candidate/aai540-product-qa-candidate-2026-10-17-01-29-09-129/output/model.tar.gz",
    "source_model_role": "existing champion",
    "monitor_run_id": "20261017_014230",
    "monitoring_receipt": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/monitoring/latest.json",
    "continual_learning_run_id": "20261017_015154",
    "continual_learning_receipt": "s3://sagemaker-us-east-1-070881885726/aai540/product-qa/continual-learning/latest.json",
    "continual_can